# Proyecto RappiPlus: de datos a decisiones de negocio

## Introducción

Este proyecto analiza el desempeño de RappiPlus a través de sus ingresos, costos, inversión en marketing y comportamiento de usuarios. El recorrido combina Python, consultas SQL y visualización en Power BI.

### Fuentes de datos

- `rappiplus_orders_raw.csv`: pedidos, precios, descuentos e ingresos.
- `rappiplus_catalog.csv`: productos, categorías y costos unitarios.
- `rappiplus_marketing_spend.csv`: inversión en marketing por canal y país.
- Tablas SQL `events`, `users` y `user_activity`: eventos, registros y actividad de usuarios.
- `experiment_checkout_ui.csv`: resultados del experimento A/B de la interfaz de checkout.

### Estructura del análisis

1. Exploración y preparación de datos.
2. Análisis de rentabilidad y ventas.
3. Análisis del embudo de conversión.
4. Retención por cohortes.
5. Evaluación estadística del experimento A/B.
6. Comunicación de resultados en Power BI.


---

## 1. Exploración y preparación de datos

### 1.1 Carga y exploración inicial

La exploración inicial presenta la estructura de los datos de pedidos, catálogo y marketing mediante muestras de registros, tipos de variables y recuentos de valores no nulos.

Los archivos se cargan en los DataFrames `orders`, `catalog` y `marketing`, utilizados en las etapas posteriores del análisis.


In [2]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [4]:
# explorar datasets
display(orders.head(25), catalog.head(8), marketing.head(5))

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28
5,order_5,user_2792,2025-04-22,mexico,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,179.34,0.0,179.34
6,order_6,user_2802,2025-03-15,Colombia,mobile,organic,Blender-XL-Red,Hogar,2.0,163.59,0.0,327.19
7,order_7,user_1083,2025-05-01,Mexico,desktop,organic,Tablet-Standard-64GB,Electronica,2.0,373.68,0.0,747.35
8,order_8,user_2352,2025-03-01,Argentina,desktop,paid_search,Laptop-Gaming-16GB,Electronica,2.0,477.27,5.0,949.54
9,order_9,user_7001,2025-05-31,Mexico,mobile,organic,Sneakers-Urban-42,Moda,1.0,339.30,5.0,334.30


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [5]:
orders.info() 
catalog.info()
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
--- 

---

### 1.2 Revisión y tratamiento de datos

Esta etapa reúne las transformaciones y exploraciones previas al cálculo de indicadores:

- Conversión de fechas en pedidos y marketing.
- Eliminación de registros duplicados en pedidos.
- Exploración de variables numéricas y exclusión de pedidos con cantidades negativas o iguales o superiores a 50.
- Exploración de variables categóricas en pedidos y marketing.
- Estandarización del formato de los nombres de países en pedidos.


In [6]:
# Convertir a fecha la columnas `fecha_hora_pedido` de orders y 'fecha' de marketing

orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

#Eliminar duplicados
orders = orders.drop_duplicates()

#Revision variables numericas
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,24950.000000,24950.000000,24950.000000,2.500000e+04
mean,7.115110,259.374527,4.502605,2.079498e+03
std,296.869961,138.690738,5.224057,9.914760e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.470000,0.000000,1.806700e+02
50%,2.000000,258.795000,0.000000,3.418050e+02
75%,2.000000,380.377500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [7]:
#Valores negativos en en cantidad, se revisaran cuantos hay
(orders['cantidad'] < 0).sum()

4

In [8]:
#eliminamos filas con cantidades negativas
orders = orders[orders['cantidad'] >= 0]

In [9]:
orders['fecha_hora_pedido'].describe()


count                   24946
unique                    181
top       2025-06-25 00:00:00
freq                      175
first     2025-01-01 00:00:00
last      2025-06-30 00:00:00
Name: fecha_hora_pedido, dtype: object

In [10]:
orders['cantidad'].value_counts().sort_index()

1.0        12345
2.0        12591
10000.0        6
20000.0        4
Name: cantidad, dtype: int64

In [11]:
# Eliminar las filas con ordenes fuera de logica
orders = orders[orders['cantidad'] < 50]

In [12]:
# explorar columnas categóricas de orders
columnas_orders = ['pais','dispositivo','fuente_referencia','nombre_producto','categoria_producto']
for col in columnas_orders:
    print(f"\n{col}")
    print(orders[col].value_counts(dropna=False))


pais
Colombia     7465
Mexico       7461
Argentina    7236
mexico        861
colombia      822
argentina     795
NaN           296
Name: pais, dtype: int64

dispositivo
desktop    12676
mobile     12240
NaN           20
Name: dispositivo, dtype: int64

fuente_referencia
social         8382
organic        8267
paid_search    8257
NaN              30
Name: fuente_referencia, dtype: int64

nombre_producto
Blender-XL-Red          4176
Vacuum-Pro-Black        4170
Jacket-Winter-M         4166
Sneakers-Urban-42       4129
Laptop-Gaming-16GB      2768
Tablet-Standard-64GB    2764
Phone-Pro-128GB         2733
NaN                       30
Name: nombre_producto, dtype: int64

categoria_producto
Hogar          8346
Moda           8295
Electronica    8265
NaN              30
Name: categoria_producto, dtype: int64


In [13]:
#Estandarizar formato de la columa 'pais'
orders['pais'] = orders['pais'].str.title()
orders['pais'].value_counts(dropna=False)

Mexico       8322
Colombia     8287
Argentina    8031
NaN           296
Name: pais, dtype: int64

In [14]:
# explorar columnas categóricas de marketing

columnas_marketing = ['pais', 'id_campaña', 'canal']
for col in columnas_marketing:
    print(f"\n{'='*40}")
    print(f"Columna: {col}")
    print(f"Valores únicos: {marketing[col].nunique(dropna=False)}")
    print(marketing[col].value_counts(dropna=False))


Columna: pais
Valores únicos: 3
Argentina    540
Colombia     540
Mexico       540
Name: pais, dtype: int64

Columna: id_campaña
Valores únicos: 9
paid_search_Argentina    180
social_Argentina         180
organic_Mexico           180
social_Colombia          180
organic_Colombia         180
social_Mexico            180
organic_Argentina        180
paid_search_Mexico       180
paid_search_Colombia     180
Name: id_campaña, dtype: int64

Columna: canal
Valores únicos: 4
paid_search    507
organic        506
social         506
NaN            101
Name: canal, dtype: int64


### 1.3 Exportación de datos

Los DataFrames resultantes se exportan como `orders_clean.csv`, `catalog_clean.csv` y `marketing_clean.csv` para su uso en la etapa de visualización.


In [15]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 2. Análisis de rentabilidad y ventas

### 2.1 Indicadores de rentabilidad

El análisis integra los pedidos, los costos unitarios del catálogo y la inversión en marketing para calcular:

- Ingresos totales (revenue).
- Costo total de productos.
- Inversión total en marketing.
- Resultado del negocio (profit), calculado como ingresos menos costos de productos e inversión en marketing.

### 2.2 Indicadores de ventas y marketing

La descripción del comportamiento comercial incluye el ticket promedio, la cantidad promedio de productos por orden, el producto con mayor cantidad vendida y la distribución del gasto de marketing por canal.


In [16]:
# INGRESO TOTAL
ingreso_total = orders['monto_total'].sum()
print(f"Ingreso total (Revenue): ${ingreso_total:,.2f}")

Ingreso total (Revenue): $9,622,281.56


In [17]:
# COSTO TOTAL
orders_costos = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')
orders_costos['costo_total'] = (orders_costos['cantidad'] * orders_costos['costo_unitario'])
costo_total = orders_costos['costo_total'].sum()
print(f"Costo Total: ${costo_total:,.2f}")

Costo Total: $3,828,869.01


In [18]:
# INVERSION TOTAL MARKETING
inversion_marketing = marketing['gasto'].sum()
print(f"Inversión total en marketing: ${inversion_marketing:,.2f}")

Inversión total en marketing: $2,871,843.53


In [19]:
# PROFIT
profit = ingreso_total - costo_total - inversion_marketing
print(f"Profit: ${profit:,.2f}")


Profit: $2,921,569.02


In [20]:
# Ticket promedio
ticket_promedio = orders['monto_total'].mean()
print(f"Ticket promedio: ${ticket_promedio:,.2f}")


Ticket promedio: $385.88


In [21]:
# Cantidad promedio de productos por orden
cantidad_promedio_productos = orders['cantidad'].mean()
print(f"Productos promedio por orden: {cantidad_promedio_productos:.2f}")

Productos promedio por orden: 1.50


In [22]:
# Producto más vendido
producto_mas_vendido = (orders.groupby('nombre_producto')['cantidad'].sum().idxmax())
print(f"Producto más vendido: {producto_mas_vendido}")

Producto más vendido: Vacuum-Pro-Black


In [23]:
# Marketing por canal
gasto_marketing_canal = marketing.groupby('canal')['gasto'].sum()
print(f"gasto_marketing_canal: {gasto_marketing_canal}")

gasto_marketing_canal: canal
organic        913533.01
paid_search    863088.21
social         918043.21
Name: gasto, dtype: float64


---

## 3. Análisis del embudo de conversión

Las consultas SQL sobre la tabla `events` permiten examinar el alcance de las etapas del proceso de compra entre los usuarios que registran una primera visita.

### 3.1 Usuarios por etapa

El recuento de usuarios únicos abarca los eventos de primera visita, selección de producto, incorporación al carrito, inicio de checkout, incorporación de información de pago y compra.

### 3.2 Conversión acumulada

Las tasas expresan el porcentaje de usuarios que alcanza cada evento respecto al total de usuarios de `first_visit`. La conversión final corresponde a la relación entre los usuarios con compra (`purchase`) y los usuarios con primera visita.


In [24]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [25]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [26]:
events['nombre_evento'].value_counts()

first_visit         29957
add_to_cart         24157
select_item         23887
begin_checkout      17971
add_payment_info    12018
purchase            12010
Name: nombre_evento, dtype: int64

In [27]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
WITH first_visit  AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'
),
add_to_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'
),
select_item AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'
),
begin_checkout AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'
),
add_payment_info AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'
),
purchase AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'purchase')
SELECT
  COUNT(fv.id_usuario) AS usuarios_first_visit,
  COUNT(ac.id_usuario) AS usuarios_add_to_cart,
  COUNT(si.id_usuario) AS usuarios_select_item,
  COUNT(bc.id_usuario) AS usuarios_begin_checkout,
  COUNT(api.id_usuario) AS usuarios_add_payment_info,
  COUNT(pu.id_usuario) AS usuarios_purchase
FROM first_visit AS fv
    LEFT JOIN add_to_cart AS ac ON fv.id_usuario = ac.id_usuario
    LEFT JOIN select_item AS si ON fv.id_usuario = si.id_usuario
    LEFT JOIN begin_checkout AS bc ON fv.id_usuario = bc.id_usuario
    LEFT JOIN add_payment_info AS api ON fv.id_usuario = api.id_usuario
    LEFT JOIN purchase AS pu ON fv.id_usuario = pu.id_usuario
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,usuarios_first_visit,usuarios_add_to_cart,usuarios_select_item,usuarios_begin_checkout,usuarios_add_payment_info,usuarios_purchase
0,7796,7439,7393,7031,6088,6068


In [28]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH first_visit  AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'
),
add_to_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'
),
select_item AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'
),
begin_checkout AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'
),
add_payment_info AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'
),
purchase AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'purchase'),
funnel_counts AS(
SELECT
  COUNT(fv.id_usuario) AS usuarios_first_visit,
  COUNT(ac.id_usuario) AS usuarios_add_to_cart,
  COUNT(si.id_usuario) AS usuarios_select_item,
  COUNT(bc.id_usuario) AS usuarios_begin_checkout,
  COUNT(api.id_usuario) AS usuarios_add_payment_info,
  COUNT(pu.id_usuario) AS usuarios_purchase
FROM first_visit AS fv
    LEFT JOIN add_to_cart AS ac ON fv.id_usuario = ac.id_usuario
    LEFT JOIN select_item AS si ON fv.id_usuario = si.id_usuario
    LEFT JOIN begin_checkout AS bc ON fv.id_usuario = bc.id_usuario
    LEFT JOIN add_payment_info AS api ON fv.id_usuario = api.id_usuario
    LEFT JOIN purchase AS pu ON fv.id_usuario = pu.id_usuario
    )

SELECT
ROUND(usuarios_add_to_cart * 100.00 / usuarios_first_visit,2) AS conversion_add_to_cart,
ROUND(usuarios_select_item * 100.00 / usuarios_first_visit,2) AS conversion_select_item,
ROUND(usuarios_begin_checkout * 100.00 / usuarios_first_visit,2) AS conversion_begin_checkout,
ROUND(usuarios_add_payment_info  * 100.00 / usuarios_first_visit,2) AS conversion_add_payment_info,
ROUND(usuarios_purchase * 100.00 / usuarios_first_visit,2) AS conversion_purchase   
FROM funnel_counts
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,conversion_add_to_cart,conversion_select_item,conversion_begin_checkout,conversion_add_payment_info,conversion_purchase
0,95.42,94.83,90.19,78.09,77.83


---

## 4. Retención por cohortes

El análisis utiliza las tablas `users` y `user_activity` para describir la actividad de los usuarios después de su registro.

### 4.1 Definición de cohortes

Las cohortes agrupan usuarios según el mes de registro. La fecha se convierte al tipo `DATE` mediante `CAST(fecha_registro AS DATE)`.

### 4.2 Actividad semanal

El recuento considera usuarios únicos con `activo = 1` en tres intervalos desde su registro:

- Semana 1: días 7 a 13 (`retenido_w1`).
- Semana 2: días 14 a 20 (`retenido_w2`).
- Semana 3: días 21 a 27 (`retenido_w3`).

### 4.3 Tasas de retención

Los porcentajes `semana_1`, `semana_2` y `semana_3` relacionan los usuarios activos de cada intervalo con el total inicial de su cohorte.


In [29]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [30]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [31]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_registro,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
),

retencion AS (
    SELECT
        c.cohorte,

        COUNT(DISTINCT c.id_usuario) AS usuarios_iniciales,

        COUNT(DISTINCT CASE
            WHEN ua.activo = 1
             AND ua.dias_despues_registro BETWEEN 7 AND 13
            THEN ua.id_usuario
        END) AS retenido_w1,

        COUNT(DISTINCT CASE
            WHEN ua.activo = 1
             AND ua.dias_despues_registro BETWEEN 14 AND 20
            THEN ua.id_usuario
        END) AS retenido_w2,

        COUNT(DISTINCT CASE
            WHEN ua.activo = 1
             AND ua.dias_despues_registro BETWEEN 21 AND 27
            THEN ua.id_usuario
        END) AS retenido_w3

    FROM cohortes c
    LEFT JOIN user_activity ua
        ON c.id_usuario = ua.id_usuario

    GROUP BY c.cohorte
)

SELECT
    cohorte,
    usuarios_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,

    ROUND(
        100.0 * retenido_w1 / NULLIF(usuarios_iniciales, 0),
        2
    ) AS semana_1,

    ROUND(
        100.0 * retenido_w2 / NULLIF(usuarios_iniciales, 0),
        2
    ) AS semana_2,

    ROUND(
        100.0 * retenido_w3 / NULLIF(usuarios_iniciales, 0),
        2
    ) AS semana_3

FROM retencion
ORDER BY cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


---

## 5. Evaluación estadística del experimento A/B

El experimento compara la tasa de conversión de compra entre los grupos de control y tratamiento para evaluar una modificación de la interfaz de checkout.

La fuente es `experiment_checkout_ui.csv`. La variable binaria `convirtio` identifica la conversión de cada observación: 1 representa una compra y 0 la ausencia de compra.

El procedimiento comprende la formulación de hipótesis, el recuento de conversiones y observaciones por grupo, la aplicación de una prueba Z de proporciones y la interpretación del valor p con un nivel de significancia de 0.05.


---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No hay evidencia suficiente de una diferencia entre las conversion con la modificacion y sin ella
   - **H₁ (Hipótesis alternativa):** hay evidencia suficiente de una diferencia entre las conversion con la modificacion y sin ella
   
**Test estadístico:** Prueba Z 
**Nivel de significancia alpha:** 0.05

In [32]:
# tu código aquí
from statsmodels.stats.proportion import proportions_ztest
df = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
df.head(10)



,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12
5,exp_user_5,control,0,mobile,Mexico,206.70,2025-01-26
6,exp_user_6,control,0,desktop,Mexico,280.11,2025-04-15
7,exp_user_7,tratamiento,0,mobile,Argentina,28.47,2025-03-18
8,exp_user_8,tratamiento,0,desktop,Mexico,287.04,2025-02-05
9,exp_user_9,control,0,mobile,Argentina,258.91,2025-02-11


In [33]:
conversiones = df.groupby('variante')['convirtio'].sum()
conversiones

variante
control        779
tratamiento    820
Name: convirtio, dtype: int64

In [34]:
totales = df.groupby('variante')['convirtio'].count()
totales

variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64

In [35]:
exitos = [conversiones['control'], conversiones['tratamiento']]
exitos

[779, 820]

In [36]:
observaciones = [totales['control'], totales['tratamiento']]
observaciones

[4965, 5035]

In [37]:
z_stat, p_value = proportions_ztest(exitos , observaciones)
print(f"Estadístico z: {z_stat}")
print(f"Valor p: {p_value}")

Estadístico z: -0.8132782986429474
Valor p: 0.41605851639119995


In [38]:
alpha = 0.05  # umbral de significancia

if p_value < alpha:
    print("Rechazamos la hipótesis nula: hay evidencia de una diferencia.")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.")

No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.


**Conclusión:** No hay evidencia estadistica suficiente para determinar una diferencia relecante con la modificacion de UI del checkout

---

## 6. Comunicación de resultados

El dashboard de Power BI complementa el notebook con una presentación visual del desempeño comercial y financiero de RappiPlus. Los archivos exportados de pedidos, catálogo y marketing constituyen la base de datos de esta etapa.

### Indicadores del negocio

- Ingresos totales.
- Profit.
- Inversión en marketing.
- Ticket promedio.
- Cantidad promedio de productos por orden.

### Perspectivas de análisis

La presentación combina una visión general de los indicadores con el análisis de la evolución temporal y la comparación por productos y categorías.


---

## Dashboard interactivo

[Consultar el dashboard de RappiPlus en Power BI](https://app.powerbi.com/view?r=eyJrIjoiODg4OWRmMmQtNzhkMC00ZWQwLWJhMDUtMmViMmY0YzhjNzIzIiwidCI6ImM1ZDFlODIzLWUyYjgtNDZiZi05MmZmLTg0ZjU0MzEzZTBhNSIsImMiOjh9&pageName=da42d321a45002ef00c2)


In [39]:
# https://app.powerbi.com/view?r=eyJrIjoiODg4OWRmMmQtNzhkMC00ZWQwLWJhMDUtMmViMmY0YzhjNzIzIiwidCI6ImM1ZDFlODIzLWUyYjgtNDZiZi05MmZmLTg0ZjU0MzEzZTBhNSIsImMiOjh9&pageName=da42d321a45002ef00c2
# link de power bi o tableau
# link de one drive / google drive